# ComfyUI Fixed Colab\n\nThis notebook installs or updates ComfyUI on Google Drive, installs ComfyUI-Manager, fixes current dependency issues such as `alembic` and stale `comfy_aimdo` imports, downloads a small SD1.5 starter model, and launches ComfyUI through Cloudflare Tunnel.\n\nRun cells from top to bottom. If an old broken `/content/drive/MyDrive/ComfyUI` exists, keep `CLEAN_STALE_FILES = True` in the setup cell.

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Install or update ComfyUI

In [ ]:
from google.colab import drive\nfrom pathlib import Path\nimport os, shutil, subprocess, sys\n\nUSE_GOOGLE_DRIVE = True\nCLEAN_STALE_FILES = True\nWORKSPACE = Path('/content/drive/MyDrive/ComfyUI') if USE_GOOGLE_DRIVE else Path('/content/ComfyUI')\n\nif USE_GOOGLE_DRIVE:\n    drive.mount('/content/drive')\n\nparent = WORKSPACE.parent\nparent.mkdir(parents=True, exist_ok=True)\n%cd {parent}\n\nif not WORKSPACE.exists():\n    !git clone https://github.com/comfyanonymous/ComfyUI.git {WORKSPACE.name}\n\n%cd {WORKSPACE}\n\n# Remove files commonly left by old/broken Colab sessions. They can shadow current ComfyUI modules.\nif CLEAN_STALE_FILES:\n    stale_paths = [\n        WORKSPACE / 'comfy_aimdo',\n        WORKSPACE / 'comfy_aimdo.py',\n        WORKSPACE / 'comfy_api_nodes',\n    ]\n    for path in stale_paths:\n        if path.is_dir():\n            shutil.rmtree(path)\n            print('Removed stale directory:', path)\n        elif path.exists():\n            path.unlink()\n            print('Removed stale file:', path)\n\n!git reset --hard HEAD\n!git pull --ff-only\n\n# Colab dependency setup. Installing requirements after git pull fixes missing alembic/sqlite deps.\n!python3 -m pip install -q --upgrade pip setuptools wheel\n!python3 -m pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121\n!python3 -m pip install -q -r requirements.txt\n!python3 -m pip install -q alembic blake3 GitPython accelerate einops transformers safetensors aiohttp pyyaml Pillow scipy tqdm psutil tokenizers torchsde kornia spandrel soundfile sentencepiece\n\nprint('ComfyUI setup complete:', WORKSPACE)

## 3. Install or update ComfyUI-Manager

In [ ]:
from pathlib import Path\nWORKSPACE = Path('/content/drive/MyDrive/ComfyUI')\ncustom_nodes = WORKSPACE / 'custom_nodes'\ncustom_nodes.mkdir(parents=True, exist_ok=True)\n%cd {custom_nodes}\n\n![ ! -d ComfyUI-Manager ] && git clone https://github.com/ltdrdata/ComfyUI-Manager.git\n%cd ComfyUI-Manager\n!git reset --hard HEAD\n!git pull --ff-only\n\n%cd {WORKSPACE}\n!python3 custom_nodes/ComfyUI-Manager/cm-cli.py restore-dependencies\nprint('ComfyUI-Manager setup complete')

## 4. Download starter models

In [ ]:
%cd /content/drive/MyDrive/ComfyUI\n!mkdir -p models/checkpoints models/vae\n!wget -c https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.ckpt -P models/checkpoints/\n!wget -c https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors -P models/vae/

## 5. Launch with Cloudflare Tunnel\n\nWait until the output shows `ComfyUI URL: https://...trycloudflare.com`. Open that link.

In [ ]:
!wget -q -P /content https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb\n!dpkg -i /content/cloudflared-linux-amd64.deb\n\nimport subprocess, threading, time, socket, re\n\ndef launch_cloudflared(port):\n    while True:\n        time.sleep(0.5)\n        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)\n        result = sock.connect_ex(('127.0.0.1', port))\n        sock.close()\n        if result == 0:\n            break\n    print('ComfyUI loaded. Starting Cloudflare Tunnel...')\n    proc = subprocess.Popen(\n        ['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{port}'],\n        stdout=subprocess.PIPE,\n        stderr=subprocess.PIPE,\n        text=True,\n    )\n    for line in proc.stderr:\n        match = re.search(r'https://[-a-zA-Z0-9.]+\\.trycloudflare\\.com', line)\n        if match:\n            print('ComfyUI URL:', match.group(0))\n\nthreading.Thread(target=launch_cloudflared, daemon=True, args=(8188,)).start()\n\n%cd /content/drive/MyDrive/ComfyUI\n!python3 main.py --dont-print-server